# B-Free x GlobalForge — Notebook 01: Setup Online (bundle builder)

**Target A: Kaggle CPU hoặc T4×2, Internet ON.** Notebook này tạo **toàn bộ assets offline** cho notebooks 02 (train) / 03 (eval), vốn chạy trên **Target B: RTX PRO 6000 Blackwell 96GB, hoàn toàn offline** (không pip internet, không git, không HF hub).

## Chuỗi pipeline (3 notebook, khép kín)

```
[01] setup-online (CPU/T4, online)
      |- wheels_rtxpro6000/   : pin stack wheels (linux x86_64, py3.11)
      |- models/vit_base_patch14_reg4_dinov2/  : DINOv2 ViT-B/14 reg4 (D5)
      |- bfree_src/            : repo P-Bao/B-Free @ integration/loss-backbone
      |- manifest.json        : versions + SHA256 + sizes
      v  (Save Version -> attach output này làm Input của 02)
[02] training (RTX PRO 6000, offline)  -> bfree_globalforge_lora_r16.pth + train_log.csv
      v  (Save Version -> attach output này làm Input của 03)
[03] eval (RTX PRO 6000, offline)      -> eval_results.csv + bar + heatmap
```

Giữa các notebook chỉ có thao tác Kaggle-native: **Save Version → attach output làm input** — không bước chuẩn bị thủ công nào khác.

**Input ngoài duy nhất** (attach thêm vào 02/03, upload thủ công một lần): B-Free training data (grip.unina.it) cho 02; baseline weights + GlobalForge assets + wild benchmarks cho 03 (liệt kê chi tiết trong từng notebook).

In [ ]:
# ============================================================
# Cell 1 - Pin versions (locked stack, plan.md)
# ============================================================
TORCH_VERSION = "2.8.0+cu128"     # RTX Pro 6000 Blackwell (sm_120) needs torch>=2.8 cu128
TORCHVISION_VERSION = "0.23.0+cu128"
TIMM_VERSION = "1.0.22"
PEFT_VERSION = "0.15.2"
TRANSFORMERS_VERSION = "4.55.4"
PANDAS_VERSION = "2.3.3"         # MUST <3 (breaks sklearn at >=3)
NUMPY_VERSION = "1.26.4"
CUDA_TAG = "cu128"
REPO_URL = "https://github.com/P-Bao/B-Free.git"
BRANCH = "integration/loss-backbone"
DINOV2_HF_REPO = "timm/vit_base_patch14_reg4_dinov2.lvd142m"

print(f"TORCH_VERSION         = {TORCH_VERSION}")
print(f"TORCHVISION_VERSION   = {TORCHVISION_VERSION}")
print(f"TIMM_VERSION          = {TIMM_VERSION}")
print(f"PEFT_VERSION          = {PEFT_VERSION}")
print(f"TRANSFORMERS_VERSION  = {TRANSFORMERS_VERSION}")
print(f"PANDAS_VERSION        = {PANDAS_VERSION}")
print(f"NUMPY_VERSION         = {NUMPY_VERSION}")
print(f"CUDA_TAG              = {CUDA_TAG}")
print(f"REPO                 = {REPO_URL} @ {BRANCH}")

## 1. Chốt version môi trường

Probe môi trường **Target A** (session này). Wheels tải ở đây phải khớp **Target B** — Kaggle dùng chung image Linux x86_64 + Python 3.11 cho mọi accelerator, nên assert Python 3.11 để chắc chắn.

In [ ]:
import platform
import subprocess
import sys

print("=" * 60)
print("Environment probe (Target A: online session)")
print("=" * 60)
print(f"python   = {platform.python_version()}")
print(f"platform = {platform.platform()}")

assert sys.version_info[:2] == (3, 11), (
    f"This session runs Python {sys.version_info[:2]} but Kaggle Target B uses 3.11 — "
    "wheels would not match. Switch the session to Python 3.11 and re-run.")

try:
    out = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
    print(out.stdout.splitlines()[2] if len(out.stdout.splitlines()) > 2 else "nvidia-smi ok")
except Exception:
    print("nvidia-smi not available (CPU session) — fine, this notebook only downloads assets.")

## 2. Tải wheel cho Target B (RTX PRO 6000)

- `torch` / `torchvision`: từ **PyTorch cu128 index**, `--no-deps` (wheel cu128 đã bundle CUDA runtime, không cần kéo `nvidia-*`/`pytorch-triton`).
- Còn lại: **`--no-deps` từng package + đầy đủ leaf dependencies** (filelock, fsspec, tokenizers, ...) để `pip install --no-index` trên Target B tự chủ hoàn toàn, không phụ thuộc version preinstall của image đích.

In [ ]:
from pathlib import Path

WHEELS = Path("/kaggle/working/wheels_rtxpro6000")
WHEELS.mkdir(parents=True, exist_ok=True)
PYTORCH_INDEX = f"https://download.pytorch.org/whl/{CUDA_TAG}"
print(f"PyTorch index: {PYTORCH_INDEX}\n")

!pip download torch=={TORCH_VERSION} \
    --index-url {PYTORCH_INDEX} --no-deps -d {WHEELS} 2>&1 | tail -3
!pip download torchvision=={TORCHVISION_VERSION} \
    --index-url {PYTORCH_INDEX} --no-deps -d {WHEELS} 2>&1 | tail -3

for p in sorted(WHEELS.glob("*.whl")):
    print(f"  {p.name:60s} {p.stat().st_size/1024/1024:8.1f} MB")

In [ ]:
# Pin stack còn lại + đầy đủ leaf deps (đủ cho pip --no-index trên Target B)
_pkgs_pinned = [
    f"timm=={TIMM_VERSION}",
    f"peft=={PEFT_VERSION}",
    f"transformers=={TRANSFORMERS_VERSION}",
    f"pandas=={PANDAS_VERSION}",
    f"numpy=={NUMPY_VERSION}",
    "matplotlib==3.11.1",
    "seaborn==0.13.2",
]
_pkgs_loose = [
    "scikit-learn", "scipy", "pyyaml", "pillow", "tqdm", "safetensors",
    "huggingface_hub", "accelerate",
    # leaf dependencies (thiếu cái nào pip --no-index sẽ fail nếu image đích không có)
    "filelock", "fsspec", "packaging", "typing-extensions", "regex",
    "requests", "certifi", "charset-normalizer", "idna", "urllib3",
    "tokenizers", "psutil", "joblib", "threadpoolctl",
    "python-dateutil", "six", "pytz", "tzdata",
    "contourpy", "cycler", "fonttools", "kiwisolver", "pyparsing",
]
for pkg in _pkgs_pinned + _pkgs_loose:
    print(f">>> pip download {pkg}")
    !pip download {pkg} --no-deps -d {WHEELS} 2>&1 | tail -2

import subprocess
n_whl = len(list(WHEELS.glob("*.whl")))
total_gb = sum(p.stat().st_size for p in WHEELS.glob("*.whl")) / 1024**3
print(f"\n{WHEELS}: {n_whl} wheels, {total_gb:.2f} GB")
assert n_whl >= 35, "wheels look incomplete (expected >=35 incl. leaf deps)"

## 3. Tải DINOv2 ViT-B/14 reg4 (backbone pretrained — D5)

Tải `model.safetensors` từ HF hub `timm/vit_base_patch14_reg4_dinov2.lvd142m` (~330MB) — Target B offline sẽ init backbone từ file này (notebook 02 resample `pos_embed` 518px → 504px khi load).

In [ ]:
import shutil
from pathlib import Path

from huggingface_hub import hf_hub_download

MODELS = Path("/kaggle/working/models")
MODELS.mkdir(parents=True, exist_ok=True)
DINO_OUT = MODELS / "vit_base_patch14_reg4_dinov2"
DINO_OUT.mkdir(exist_ok=True)

dst = DINO_OUT / "model.safetensors"
if dst.exists():
    print(f"[skip] already at {dst}")
else:
    src = hf_hub_download(repo_id=DINOV2_HF_REPO, filename="model.safetensors")
    shutil.copyfile(src, dst)
print(f"saved: {dst} ({dst.stat().st_size/1024**2:.1f} MB)")

from safetensors.torch import load_file

sd = load_file(str(dst))
pe = sd["pos_embed"]
assert sd["patch_embed.proj.weight"].shape[0] == 768, "not a ViT-B checkpoint"
assert tuple(pe.shape) == (1, 37 * 37 + 5, 768), (
    f"unexpected pos_embed {tuple(pe.shape)} — expected 518px grid (37x37) + 5 prefix tokens")
print(f"OK: {len(sd)} tensors | embed_dim=768 | pos_embed {tuple(pe.shape)} (518px grid + 5 prefix)")

## 4. Copy B-Free repo source (branch `integration/loss-backbone`)

Target B offline không git clone được → đóng gói source vào bundle (bỏ `.git`, `__pycache__`). Sau đó **smoke test trên CPU** để xác nhận bundle model (backbone + real LIB/GSR + L_DCS) import + forward + `compute_loss` + backward chạy đúng — đây là verify cuối cùng trước khi vào offline.

In [ ]:
import glob
import shutil
import subprocess
import sys
from pathlib import Path

BFREE_SRC = Path("/kaggle/working/bfree_src")
if BFREE_SRC.exists():
    shutil.rmtree(BFREE_SRC)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(BFREE_SRC)], check=True)

shutil.rmtree(BFREE_SRC / ".git", ignore_errors=True)
for p in BFREE_SRC.rglob("__pycache__"):
    shutil.rmtree(p, ignore_errors=True)

assert (BFREE_SRC / "code/networks/bfree_globalforge_vit.py").is_file(), "clone failed: backbone file missing"
assert not glob.glob(str(BFREE_SRC / "code/modules/*_stub.py")), "stub files present — K0 not merged?"
n_py = len(list(BFREE_SRC.rglob("*.py")))
print(f"OK: {BFREE_SRC} ({n_py} .py files, .git removed, no stubs — K0 verified)")

In [ ]:
# Smoke test tren CPU: import + forward + compute_loss + backward (224px)
import sys

sys.path.insert(0, str(BFREE_SRC / "code"))

import torch

from networks.bfree_globalforge_vit import BFreeGlobalForgeViT

model = BFreeGlobalForgeViT(img_size=224, pretrained=False)
x1 = torch.randn(2, 3, 224, 224)
x2 = torch.randn(2, 3, 224, 224)
y = torch.tensor([0, 1])
with torch.no_grad():
    out = model(x1)
assert out["logits"].shape == (2, 2) and out["cls"].shape == (2, 768)
total, ce, dcs = model.compute_loss(x1, x2, y)
total.backward()
print(f"SMOKE OK | logits {tuple(out['logits'].shape)} | total={float(total):.4f} "
      f"ce={float(ce):.4f} dcs={float(dcs):.4f} | backward OK")

## 5. Manifest + tổng kết

Ghi `manifest.json` (versions + SHA256 từng file + size từng component) — Target B dùng để kiểm tra tính toàn vẹn của bundle sau khi attach.

In [ ]:
import hashlib
import json
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path

WORK = Path("/kaggle/working")


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "cuda_tag_for_rtxpro6000": CUDA_TAG,
    "repo": {"url": REPO_URL, "branch": BRANCH},
    "dinov2_hf_repo": DINOV2_HF_REPO,
    "pins": {
        "torch": TORCH_VERSION, "torchvision": TORCHVISION_VERSION,
        "timm": TIMM_VERSION, "peft": PEFT_VERSION,
        "transformers": TRANSFORMERS_VERSION, "pandas": PANDAS_VERSION,
        "numpy": NUMPY_VERSION, "matplotlib": "3.11.1", "seaborn": "0.13.2",
    },
    "components": {},
}

for sd in ["wheels_rtxpro6000", "models", "bfree_src"]:
    root = WORK / sd
    if not root.exists():
        continue
    files = sorted(p for p in root.rglob("*") if p.is_file())
    manifest["components"][sd] = {
        "file_count": len(files),
        "total_mb": sum(p.stat().st_size for p in files) / 1024**2,
        "sha256": {str(p.relative_to(WORK)): sha256_file(p) for p in files},
    }

out = WORK / "manifest.json"
out.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"[OK] {out} ({out.stat().st_size/1024:.1f} KB)\n")
print(f"{'component':<22} {'files':>7} {'size (MB)':>10}")
print("-" * 45)
for name, info in manifest["components"].items():
    print(f"{name:<22} {info['file_count']:>7} {info['total_mb']:>10.1f}")

## Bundle Ready (K1 pass)

`/kaggle/working/` giờ chứa **đầy đủ assets offline**:
- `wheels_rtxpro6000/` — pin stack + leaf deps (linux x86_64, py3.11)
- `models/vit_base_patch14_reg4_dinov2/model.safetensors` — DINOv2 ViT-B/14 reg4 (D5)
- `bfree_src/` — repo `integration/loss-backbone` (đã smoke test)
- `manifest.json` — versions + SHA256

**Bước giao tiếp giữa notebook (Kaggle-native, bắt buộc):**
1. **Save Version → Save & Run All (Commit)** để chốt output.
2. Mở notebooks 02/03 → **Add Input → Your Work → notebook 01 này** (output của nó thành input `/kaggle/input/<slug>/...` chứa `wheels_rtxpro6000/`, `bfree_src/`, `models/`).
3. 02 cần thêm input ngoài: **B-Free training data** (upload dataset có `COCO_real_512/` + 6 thư mục `SD2.1_*/`).
4. 03 cần thêm: **K2 output** (notebook 02), **B-Free baseline weights**, **GlobalForge assets**, **wild benchmarks** (chi tiết trong notebook 03).